In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
from pathlib import Path

notebook_path = "/u/skarmakar1/version_check/llm_steering-main/sk"
sys.path.append(notebook_path)

In [3]:
import torch
import numpy as np

from inversion_utils import *
import pickle
from sklearn.model_selection import train_test_split

In [4]:
SEED = 0
# SEED = 1

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
np.random.seed(SEED)

torch.backends.cudnn.benchmark = True 
torch.backends.cuda.matmul.allow_tf32 = True

LLM = namedtuple('LLM', ['language_model', 'tokenizer', 'processor', 'name', 'model_type'])

In [5]:
model_type = 'llama'
MODEL_VERSION='3.1'
MODEL_SIZE='8B'

# model_type = 'gemma'
# MODEL_VERSION='3'
# MODEL_SIZE='1B'
# MODEL_SIZE='12B'

# model_type = 'qwen'
# MODEL_VERSION='3'
# MODEL_SIZE='4B'
# # MODEL_SIZE='8B'

llm = select_llm(model_type, MODEL_VERSION=MODEL_VERSION, MODEL_SIZE=MODEL_SIZE)

Loading meta-llama/Meta-Llama-3.1-8B-Instruct


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [6]:
hidden_layers = list(range(-llm.language_model.config.num_hidden_layers+1, 0))
print(hidden_layers)

[-31, -30, -29, -28, -27, -26, -25, -24, -23, -22, -21, -20, -19, -18, -17, -16, -15, -14, -13, -12, -11, -10, -9, -8, -7, -6, -5, -4, -3, -2, -1]


Training

In [7]:
all_authors = ["hemingway", "faulkner", "joyce", "austen", "woolf", "kafka", "stein", "cummings", "nabokov", "wilde", "twain", "wodehouse", "mccarthy", "carver", "bukowski", "márquez", "borges", "thompson", "lovecraft", "pratchett",]

In [8]:
orig_author = 'kafka'
target_author = 'hemingway'
# test_set = ['faulkner',]
test_set = ['joyce',]

source_shift = True
# source_shift = False

other_authors = [element for element in all_authors if element not in [orig_author, target_author] + test_set]

path = '../all_gitignore/xRFM/style_llama/'

X = read_single_translate(llm, orig_author, other_authors, path, source_shift=source_shift, hidden_layers=hidden_layers)
Y = read_single_translate(llm, target_author, other_authors, path, source_shift=source_shift, hidden_layers=hidden_layers)

Source Constant
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Using xRFM library for RFM-based steering vector extraction
Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Using xRFM library for RFM-based steering vector extraction
Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameter

In [9]:
print(X[-2].shape)
print(Y[-2].shape)

torch.Size([17, 4096])
torch.Size([17, 4096])


In [10]:
big_X = torch.vstack(list(X.values())).cpu().numpy()
big_Y = torch.vstack(list(Y.values())).cpu().numpy()

In [11]:
print(big_X.shape)
print(big_Y.shape)

(527, 4096)
(527, 4096)


In [12]:
from sklearn.utils import shuffle

big_X_shuffled, big_Y_shuffled = shuffle(big_X, big_Y, random_state=SEED)

In [13]:
# model_lrr = LRR_single(big_X_shuffled, big_Y_shuffled, alpha=1000.0, print_error=True)
model_lrr = LRR_single(big_X_shuffled, big_Y_shuffled, alpha=np.arange(1, 4), print_error=True)

lrr_models = {i: model_lrr for i in hidden_layers}

Running with alpha: [  10.  100. 1000.]
Best lambda: 100.0
Training RMSE: 0.0008, Training R2: 0.9969
Done.


In [14]:
if source_shift:
    with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/style/llama8b/source_{orig_author}TO{target_author}-{test_set}.pkl', 'wb') as file:
        pickle.dump(lrr_models, file)
else:
    with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/style/llama8b/dest_{orig_author}TO{target_author}-{test_set}.pkl', 'wb') as file:
        pickle.dump(lrr_models, file)

Testing

In [15]:
author_map = {
        "hemingway": "Ernest Hemingway",
        "faulkner": "William Faulkner",
        "joyce": "James Joyce",
        "austen": "Jane Austen",
        "woolf": "Virginia Woolf",
        "kafka": "Franz Kafka",
        "stein": "Gertrude Stein",
        "cummings": "E.E. Cummings",
        "nabokov": "Vladimir Nabokov",
        "wilde": "Oscar Wilde",
        "twain": "Mark Twain",
        "wodehouse": "P.G. Wodehouse",
        "mccarthy": "Cormac McCarthy",
        "carver": "Raymond Carver",
        "bukowski": "Charles Bukowski",
        "márquez": "Gabriel García Márquez",
        "borges": "Jorge Luis Borges",
        "thompson": "Hunter S. Thompson",
        "lovecraft": "H.P. Lovecraft",
        "pratchett": "Terry Pratchett",
    }

In [20]:
prompts = ["How does a toaster work?"] # "hemingway"

# style = "kafka"
# style = "hemingway"
style = "faulkner"
# style = "joyce"

sent = 'Write a response to the following statement in the literary style of {}. \nStatement: {}'.format(author_map[style], prompts[0])

messages = [
    {"role": "user", "content": sent}
]

# prompts = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
tok_prompts = llm.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True).strip()

out = llm.tokenizer(tok_prompts, return_tensors='pt', padding=True, add_special_tokens=False).to(llm.language_model.device)

outputs = llm.language_model.generate(**out, max_new_tokens=200, do_sample=False)
print(llm.tokenizer.decode(outputs[0], skip_special_tokens=True))

system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

Write a response to the following statement in the literary style of William Faulkner. 
Statement: How does a toaster work?assistant

"The toaster, a contraption of twisted metal and mystifying intent, stood before us, its very existence a testament to the ingenuity of a world that had long since abandoned the simple, the pure, and the unadulterated. And yet, we found ourselves drawn to it, like moths to the flame that it so callously dispensed, our curiosity a siren's song that beckoned us to unravel the tangled threads of its operation.

"It was a dance of electrons, a waltz of heat and light, a symphony of metal and wire that culminated in the perfect, golden-brown toast, a culinary epiphany that defied the cruel whims of the universe. The toaster, a device of precision and artistry, harnessed the raw power of the electric current, channeling it through a labyrinthine network of coils and resistors, until i

Testing Source Shift

In [7]:
# Loading

orig_author = 'kafka'
target_author = 'hemingway'
test_set = ['faulkner',]

with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/style/llama8b/source_{orig_author}TO{target_author}-{test_set}.pkl', 'rb') as file:
# with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/language/qwen/source_{orig_lang}TO{target_lang}-{test_set}.pkl', 'rb') as file:
    lrr_models = pickle.load(file)

In [8]:
# coef = 0.4
# coef = 0.45
coef = 0.5
# coef = 0.55
# coef = 0.65
# coef = 0.7

max_tokens = 200

path = "../all_gitignore/xRFM/style_llama/"

orig_author = 'kafka'

# partial = "The clock, a monolith of precision, a sentinel of time, its workings a mystery that has long fascinated and confounded me.  As I stand before its imposing face, I am struck by the futility of my inquiry.  How does a clock work?  The question itself seems a cruel jest, a taunt from the very fabric of existence. I reach out a trembling hand, as if to grasp the elusive threads of its mechanism, but they slip through my fingers like sand."

partial = "The toaster, a contraption of bewildering complexity, a device that defies comprehension, and yet, it stands before us, a monolith of functionality, its purpose clear, yet its inner workings a labyrinth of mystery. As I stand before the toaster, its metal exterior gleaming in the faint light of the kitchen, I am struck by the sense of futility that pervades my existence. How can I possibly hope to grasp the intricacies of this device, when the very fabric of reality seems to conspire against me? And yet, I am drawn to the toaster, like a moth to the flame, helpless to resist its siren call."

stylised_prompt = [f"Continue the following story, while maitaining the literary style:\n {partial}"]

a = 'faulkner'

a_controller = load_controller_translate(llm, a, orig_author, path=path)
orig_a = a_controller.directions
out = test_concept_vector(a_controller, concept=f"coef: {coef}, ques original: {orig_author}, steered: {a}", prompts=stylised_prompt, coef=coef, max_tokens=max_tokens)


# new_partial = "The clock. A machine of gears and springs, ticking away like a heartbeat. It's simple, really. A mainspring, coiled tight, stores energy. A series of gears, meshing together, release it. The balance wheel, a delicate dance, regulates the flow." # hemingway

new_partial = "The toaster. A simple thing, really. A box with a lever, a heating coil, and a spring. You put bread in, press the lever down, and the coil heats up. The spring pushes the bread up against the coil, and the heat sears it. It's a straightforward thing, no fuss, no muss. The bread gets toasted, and you're done. That's it. No need to overthink it. Just a machine doing its job, like a man doing his work." # hemingway

new_stylised_prompt = [f"Continue the following story, while maitaining the literary style:\n {new_partial}"]

# out = test_concept_vector(a_controller, concept=f"coef: {coef}, ques original: {target_author}, steered: {a}", prompts=new_stylised_prompt, coef=coef, max_tokens=max_tokens)

a1_controller = load_controller_translate(llm, a, target_author, path=path)
orig_a1 = a1_controller.directions
out = test_concept_vector(a1_controller, concept=f"coef: {coef}, ques original: {target_author}, steered: {a}", prompts=new_stylised_prompt, coef=coef, max_tokens=max_tokens)


trans_a = apply_auto(orig_a, lrr_models)
a_controller.directions = trans_a

out = test_concept_vector(a_controller, concept=f"coef: {coef}, ques original: {target_author}, source_M({orig_author}to{target_author})(steered): {a}", prompts=new_stylised_prompt, coef=coef, max_tokens=max_tokens, orig=False)

Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Using xRFM library for RFM-based steering vector extraction
Detector found

========================== No Control ==========================
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

Continue the following story, while maitaining the literary style:
 The toaster, a contraption of bewildering complexity, a device that defies comprehension, and yet, it stands before us, a monolith of functionality, its purpose clear, yet its inner workings a labyrinth of mystery. As I stand before the toaster, its metal exterior gleaming in the faint light

In [9]:
trans_a = apply_auto(orig_a, lrr_models)
a_controller.directions = trans_a

out = test_concept_vector(a_controller, concept=f"coef: {coef}, ques original: {target_author}, source_M({orig_author}to{target_author})(steered): {a}", prompts=new_stylised_prompt, coef=coef, max_tokens=max_tokens, orig=False)


========================== + coef: 0.5, ques original: hemingway, source_M(kafkatohemingway)(steered): faulkner Control (normal) ==========================
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

Continue the following story, while maitaining the literary style:
 The toaster. A simple thing, really. A box with a lever, a heating coil, and a spring. You put bread in, press the lever down, and the coil heats up. The spring pushes the bread up against the coil, and the heat sears it. It's a straightforward thing, no fuss, no muss. The bread gets toasted, and you're done. That's it. No need to overthink it. Just a machine doing its job, like a man doing his work.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

And yet, as the hours tick by, the toaster's gentle hum becomes a lullaby of sorts, a soothing melody that weaves itself into the very fabr

Testing Target Shift

In [10]:
# Loading

orig_author = 'kafka'
target_author = 'hemingway'
test_set = ['faulkner',]

with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/style/llama8b/dest_{orig_author}TO{target_author}-{test_set}.pkl', 'rb') as file:
# with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/language/qwen/dest_{orig_lang}TO{target_lang}-{test_set}.pkl', 'rb') as file:
    lrr_models = pickle.load(file)

In [11]:
# coef = 0.4
# coef = 0.45
coef = 0.5
# coef = 0.55
# coef = 0.65
# coef = 0.7

max_tokens = 200

path = "../all_gitignore/xRFM/style_llama/"

orig_author = 'kafka'

# partial = "The clock, a monolith of measured time, its face a mask of precision, conceals within its mechanical heart a labyrinth of gears and springs, a dance of metal and oil, a symphony of ticking and tocking that echoes through the chambers of our minds like the beat of a distant drum. It is a device of human ingenuity, a contraption born of our insatiable quest for order and control, a desperate attempt to impose meaning on the chaotic flux of existence."

partial = "The toaster, a contraption of bewildering complexity, a device that defies comprehension, and yet, it stands before us, a monolith of functionality, its purpose clear, yet its inner workings a labyrinth of mystery. As I stand before the toaster, its metal exterior gleaming in the faint light of the kitchen, I am struck by the sense of futility that pervades my existence. How can I possibly hope to grasp the intricacies of this device, when the very fabric of reality seems to conspire against me? And yet, I am drawn to the toaster, like a moth to the flame, helpless to resist its siren call."

stylised_prompt = [f"Continue the following story, while maitaining the style:\n {partial}"]

a = 'faulkner'

a_controller = load_controller_translate(llm, orig_author, a, path=path)
orig_a = a_controller.directions
out = test_concept_vector(a_controller, concept=f"coef: {coef}, ques original: {a}, steered: {orig_author}", prompts=stylised_prompt, coef=coef, max_tokens=max_tokens)


a1_controller = load_controller_translate(llm, target_author, a, path=path)
orig_a1 = a1_controller.directions
out = test_concept_vector(a1_controller, concept=f"coef: {coef}, ques original: {a}, steered: {target_author}", prompts=stylised_prompt, coef=coef, max_tokens=max_tokens)


trans_a = apply_auto(orig_a, lrr_models)
a_controller.directions = trans_a

out = test_concept_vector(a_controller, concept=f"coef: {coef}, ques original: {a}, dest_M({orig_author}to{target_author})(steered): {target_author}", prompts=stylised_prompt, coef=coef, max_tokens=max_tokens, orig=False)

Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Using xRFM library for RFM-based steering vector extraction
Detector found

========================== No Control ==========================
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

Continue the following story, while maitaining the style:
 The toaster, a contraption of bewildering complexity, a device that defies comprehension, and yet, it stands before us, a monolith of functionality, its purpose clear, yet its inner workings a labyrinth of mystery. As I stand before the toaster, its metal exterior gleaming in the faint light of the k